In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
import warnings

warnings.filterwarnings('ignore')

def executar_backtest_financeiro_min_var(diretorio_dados, custo_corretagem_spread=0.005):
    print("=== INÍCIO DO BACKTEST FINANCEIRO (JANELAS EXPANSIVAS L1 - MÍNIMA VARIÂNCIA) ===")
    
    print("1. A carregar a base mestre consolidada em CSV...")
    caminho_mestre = os.path.join(diretorio_dados, "base_mestre_consolidada_IBOV_CDI_SELIC.csv")
    df_mestre = pd.read_csv(caminho_mestre, sep=';', decimal=',', encoding='latin1')
    
    # Blindagem Datas Mestre
    coluna_data = df_mestre.columns[0]
    df_mestre[coluna_data] = pd.to_datetime(df_mestre[coluna_data], errors='coerce')
    df_mestre.set_index(coluna_data, inplace=True)
    df_mestre.index.name = 'Data'
    df_mestre.index = df_mestre.index.normalize()
    if df_mestre.index.tz is not None:
        df_mestre.index = df_mestre.index.tz_localize(None)
    df_mestre = df_mestre.sort_index()
    df_mestre = df_mestre[~df_mestre.index.duplicated(keep='last')]
        
    for coluna in df_mestre.columns:
        if df_mestre[coluna].dtype == 'object' or df_mestre[coluna].dtype.name in ['string', 'string[pyarrow]']:
            df_mestre[coluna] = df_mestre[coluna].astype(str).str.replace(',', '.')
        df_mestre[coluna] = pd.to_numeric(df_mestre[coluna], errors='coerce')
    
    colunas_macro = ['IBOV', 'CDI', 'SELIC']
    colunas_ativos = [col for col in df_mestre.columns if col not in colunas_macro]
    
    print("2. A calcular os retornos diários e alinhar as taxas...")
    df_retornos_ativos = df_mestre[colunas_ativos].pct_change().dropna(how='all')
    retornos_ibov = df_mestre['IBOV'].pct_change().dropna(how='all')
    retornos_cdi = df_mestre['CDI'].loc[df_retornos_ativos.index].fillna(0)
    
    # --- PUXAR O FICHEIRO DE MÍNIMA VARIÂNCIA ---
    print("3. A carregar o histórico de pesos da otimização defensiva...")
    caminho_pesos = os.path.join(diretorio_dados, "historico_pesos_l1_5anos_M_min_var.parquet")
    df_pesos = pd.read_parquet(caminho_pesos)
    
    # Blindagem Pesos
    if not isinstance(df_pesos.index, pd.DatetimeIndex):
        coluna_data_pesos = df_pesos.columns[0]
        df_pesos[coluna_data_pesos] = pd.to_datetime(df_pesos[coluna_data_pesos], errors='coerce')
        df_pesos.set_index(coluna_data_pesos, inplace=True)
    df_pesos.index.name = 'Data'
    df_pesos.index = df_pesos.index.normalize()
    if df_pesos.index.tz is not None:
        df_pesos.index = df_pesos.index.tz_localize(None)
    df_pesos = df_pesos.sort_index()
    df_pesos = df_pesos[~df_pesos.index.duplicated(keep='last')]
    for coluna in df_pesos.columns:
        if df_pesos[coluna].dtype == 'object' or df_pesos[coluna].dtype.name in ['string', 'string[pyarrow]']:
            df_pesos[coluna] = df_pesos[coluna].astype(str).str.replace(',', '.')
        df_pesos[coluna] = pd.to_numeric(df_pesos[coluna], errors='coerce')
    
    # 4. Alinhamento Cronológico e Custos
    print("4. A cruzar a alocação do modelo com os retornos reais do mercado...")
    data_inicio_backtest = df_pesos.index[0]
    
    df_retornos_ativos = df_retornos_ativos.loc[data_inicio_backtest:]
    retornos_ibov = retornos_ibov.loc[data_inicio_backtest:]
    retornos_cdi = retornos_cdi.loc[data_inicio_backtest:]
    
    pesos_diarios = df_pesos.reindex(df_retornos_ativos.index).ffill().shift(1)
    pesos_diarios = pesos_diarios.dropna(how='all')
    
    df_retornos_ativos = df_retornos_ativos.loc[pesos_diarios.index]
    retornos_ibov = retornos_ibov.loc[pesos_diarios.index]
    retornos_cdi = retornos_cdi.loc[pesos_diarios.index]
    
    retorno_bruto_portfolio = (pesos_diarios * df_retornos_ativos).sum(axis=1)
    giro_diario = pesos_diarios.diff().abs().sum(axis=1).fillna(0)
    custo_transacao = giro_diario * custo_corretagem_spread
    retorno_liquido_portfolio = retorno_bruto_portfolio - custo_transacao
    
    patrimonio_portfolio = (1 + retorno_liquido_portfolio).cumprod() - 1
    patrimonio_ibov = (1 + retornos_ibov).cumprod() - 1
    patrimonio_cdi = (1 + retornos_cdi).cumprod() - 1
    
    df_patrimonio = pd.DataFrame({
        'Carteira_L1_Otimizada': patrimonio_portfolio,
        'IBOV_Mercado': patrimonio_ibov,
        'CDI_RendaFixa': patrimonio_cdi
    })
    
    print("5. A renderizar o Gráfico da Vitória...")
    plt.figure(figsize=(14, 7))
    plt.plot(df_patrimonio.index, df_patrimonio['Carteira_L1_Otimizada'], label='Janelas Expansivas + L1 (Mínima Variância)', color='dodgerblue', linewidth=2)
    plt.plot(df_patrimonio.index, df_patrimonio['IBOV_Mercado'], label='IBOVESPA', color='red', linestyle='--', linewidth=1.5)
    plt.plot(df_patrimonio.index, df_patrimonio['CDI_RendaFixa'], label='CDI', color='green', linestyle=':', linewidth=2)
    
    plt.title('Performance Backtest: Mínima Variância (Janelas Expansivas com L1)', fontsize=16)
    plt.xlabel('Ano', fontsize=12)
    plt.ylabel('Retorno Acumulado (%)', fontsize=12)
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    plt.legend(loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3)
    
    # --- NOVO NOME DA IMAGEM ---
    caminho_grafico = os.path.join(diretorio_dados, "equity_curve_l1_min_var.png")
    plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
    plt.close()
    
    ret_total_port = df_patrimonio['Carteira_L1_Otimizada'].iloc[-1]
    ret_total_ibov = df_patrimonio['IBOV_Mercado'].iloc[-1]
    ret_total_cdi = df_patrimonio['CDI_RendaFixa'].iloc[-1]
    
    print("\n=== RESUMO FINANCEIRO (OUT-OF-SAMPLE) ===")
    print(f"Retorno Acumulado Carteira MinVar: {ret_total_port:.2%}")
    print(f"Retorno Acumulado IBOVESPA:        {ret_total_ibov:.2%}")
    print(f"Retorno Acumulado CDI:             {ret_total_cdi:.2%}")
    print(f"Gráfico exportado para: {caminho_grafico}")
    print("=========================================")
    
    return df_patrimonio

pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
df_final = executar_backtest_financeiro_min_var(pasta_dados)

=== INÍCIO DO BACKTEST FINANCEIRO (JANELAS EXPANSIVAS L1 - MÍNIMA VARIÂNCIA) ===
1. A carregar a base mestre consolidada em CSV...
2. A calcular os retornos diários e alinhar as taxas...
3. A carregar o histórico de pesos da otimização defensiva...
4. A cruzar a alocação do modelo com os retornos reais do mercado...
5. A renderizar o Gráfico da Vitória...

=== RESUMO FINANCEIRO (OUT-OF-SAMPLE) ===
Retorno Acumulado Carteira MinVar: 460.20%
Retorno Acumulado IBOVESPA:        215.00%
Retorno Acumulado CDI:             166.14%
Gráfico exportado para: C:\VSCodeWorkspace\TCC_Escrito\data\equity_curve_l1_min_var.png
